# sgd-vanilla-from-scratch — faded example 3: Run multiple SGD steps and collect loss history

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sgd-vanilla-from-scratch`. The last cell reports your progress on the `Optimizer: SGD vanilla from scratch` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: SGD vanilla from scratch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sgd-vanilla-from-scratch`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sgd-vanilla-from-scratch"
DD_SUBTOPIC = "Optimizer: SGD vanilla from scratch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A training loop applies SGD repeatedly: compute loss, compute gradient, call sgd_step, record the loss. The loss should be recorded before the step so the log reads [L(w0), L(w1), ...], i.e., the loss at each parameter value before that step's update.

## Faded exercise 3

Implement `run_sgd(w0, w_star, lr, n_steps)` that optimizes the quadratic loss `0.5 * (w - w_star)^2` for `n_steps` steps.

1. Initialize w as a MiniTensor at w0.
2. For each step: compute loss, record it, set grad, call sgd_step.
3. Return (final_w_value, losses_list).

The blank step is computing the loss value before each SGD step.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        p.grad = None

def run_sgd(w0, w_star, lr, n_steps):
    w = MiniTensor(t.tensor([float(w0)]), requires_grad=True)
    losses = []
    for _ in range(n_steps):
        raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
        w.grad = w.array - w_star
        sgd_step([w], lr)
    return w.array.item(), losses

final_w, losses = run_sgd(4.0, 1.0, 0.1, 40)
print(f'Final w: {final_w:.4f}  (near 1.0)')
print(f'First 5 losses: {[round(l, 4) for l in losses[:5]]}')
print(f'Monotone: {all(losses[i] >= losses[i+1] for i in range(len(losses)-1))}')


def _test():
    import torch as t

    class MiniTensor:
        def __init__(self, array, requires_grad=False):
            self.array = array
            self.requires_grad = requires_grad
            self.grad = None
            self.recipe = None

    def sgd_step(params, lr):
        for p in params:
            if p.grad is None:
                continue
            p.array -= lr * p.grad
            p.grad = None

    final_w, losses = run_sgd(4.0, 1.0, 0.1, 40)
    assert len(losses) == 40, f'expected 40 losses, got {len(losses)}'
    expected_loss_0 = 0.5 * (4.0 - 1.0) ** 2
    assert abs(losses[0] - expected_loss_0) < 1e-5, f'losses[0]={losses[0]}'
    assert all(losses[i] >= losses[i+1] - 1e-9 for i in range(len(losses)-1)), 'not monotone'
    assert abs(final_w - 1.0) < 0.3, f'final_w={final_w} not converging toward 1.0'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.grad = None
        self.recipe = None

def sgd_step(params, lr):
    for p in params:
        if p.grad is None:
            continue
        p.array -= lr * p.grad
        p.grad = None

def run_sgd(w0, w_star, lr, n_steps):
    w = MiniTensor(t.tensor([float(w0)]), requires_grad=True)
    losses = []
    for _ in range(n_steps):
        loss = 0.5 * (w.array.item() - w_star) ** 2
        losses.append(loss)
        w.grad = w.array - w_star
        sgd_step([w], lr)
    return w.array.item(), losses

final_w, losses = run_sgd(4.0, 1.0, 0.1, 40)
print(f'Final w: {final_w:.4f}  (near 1.0)')
print(f'First 5 losses: {[round(l, 4) for l in losses[:5]]}')
print(f'Monotone: {all(losses[i] >= losses[i+1] for i in range(len(losses)-1))}')
```
</details>